# NovaTech — Parte 2: Generación de datos sintéticos

Genero datos coherentes con `Faker` para las 8 tablas y los cargo en BigQuery, respetando el orden y la integridad referencial (los IDs que uso como FK siempre existen en la tabla padre).

Volúmenes usados (alineados con la guía de costes — muy por debajo del Free Tier de BigQuery):
- 8 categorías, 70 productos
- 500 clientes
- 2000 pedidos (~1-4 líneas cada uno)

**Antes de este notebook ya he ejecutado `01_setup_bigquery.ipynb`** (dataset y tablas ya creadas).

In [1]:
import os
import random
from datetime import datetime, timedelta

import pandas as pd
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from faker import Faker
from google.cloud import bigquery
from google.oauth2 import service_account

dotenv_path = find_dotenv()
load_dotenv(dotenv_path)
PROJECT_ROOT = Path(dotenv_path).parent if dotenv_path else Path.cwd()
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

fake = Faker()
Faker.seed(RANDOM_SEED)

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID", "novatech")
CREDENTIALS_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

credentials = service_account.Credentials.from_service_account_file(PROJECT_ROOT / CREDENTIALS_PATH)
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
print(f"Cliente listo -> {PROJECT_ID}.{DATASET_ID}")

c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Cliente listo -> proyecto-507914.novatech


c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.cloud.bigquery once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.bigquery past that date.
  warnings.warn(message, FutureWarning)


## Parámetros de volumen

In [2]:
N_CUSTOMERS = 500
N_PRODUCTS = 70
N_ORDERS = 2000

COUNTRIES = ['Spain', 'France', 'Germany', 'Italy', 'Portugal', 'Netherlands']
COUNTRY_WEIGHTS = [0.40, 0.15, 0.15, 0.12, 0.10, 0.08]

CHANNELS = ['organic', 'paid_ads', 'social_media', 'referral', 'email_marketing']
CHANNEL_WEIGHTS = [0.30, 0.25, 0.20, 0.15, 0.10]

ORDER_STATUSES = ['delivered', 'shipped', 'processing', 'pending', 'cancelled']
ORDER_STATUS_WEIGHTS = [0.65, 0.12, 0.08, 0.08, 0.07]

PAYMENT_METHODS = ['credit_card', 'paypal', 'bank_transfer']

TODAY = datetime(2025, 1, 1)
ONE_YEAR_AGO = TODAY - timedelta(days=365)

## 1. `categories`

In [3]:
CATEGORY_DEFS = [
    ("Smartphones", "Telefonos moviles y accesorios de conectividad"),
    ("Laptops", "Portatiles para uso personal, gaming y profesional"),
    ("Audio", "Auriculares, altavoces y equipos de sonido"),
    ("Wearables", "Relojes inteligentes y pulseras de actividad"),
    ("Tablets", "Tablets y accesorios de escritura/dibujo digital"),
    ("Gaming", "Consolas, mandos y perifericos de videojuegos"),
    ("Accesorios", "Cables, fundas, cargadores y complementos"),
    ("Fotografia", "Camaras, drones y accesorios fotograficos"),
]

df_categories = pd.DataFrame([
    {"category_id": i + 1, "name": name, "description": desc}
    for i, (name, desc) in enumerate(CATEGORY_DEFS)
])
df_categories

,category_id,name,description
0,1,Smartphones,Telefonos moviles y accesorios de conectividad
1,2,Laptops,"Portatiles para uso personal, gaming y profesi..."
2,3,Audio,"Auriculares, altavoces y equipos de sonido"
3,4,Wearables,Relojes inteligentes y pulseras de actividad
4,5,Tablets,Tablets y accesorios de escritura/dibujo digital
5,6,Gaming,"Consolas, mandos y perifericos de videojuegos"
6,7,Accesorios,"Cables, fundas, cargadores y complementos"
7,8,Fotografia,"Camaras, drones y accesorios fotograficos"


## 2. `products`

In [4]:
PRODUCT_WORDS = {
    "Smartphones": ["Phone", "Mobile", "Edge", "Nova", "Pulse"],
    "Laptops": ["Book", "Pro", "Air", "Flex", "Studio"],
    "Audio": ["Buds", "Sound", "Beat", "Wave", "Tone"],
    "Wearables": ["Watch", "Band", "Fit", "Track", "Pulse"],
    "Tablets": ["Tab", "Pad", "Slate", "Canvas", "Note"],
    "Gaming": ["Play", "Pad", "Controller", "Arcade", "Console"],
    "Accesorios": ["Cable", "Case", "Charger", "Mount", "Hub"],
    "Fotografia": ["Cam", "Lens", "Shot", "Drone", "Frame"],
}
BRANDS = ["Zenlo", "Kortex", "Vantix", "Orbeon", "Nyra", "Halion", "Drakon", "Ecliptic"]

products_rows = []
for pid in range(1, N_PRODUCTS + 1):
    category = df_categories.sample(1, random_state=pid).iloc[0]
    word = random.choice(PRODUCT_WORDS[category["name"]])
    brand = random.choice(BRANDS)
    name = f"{brand} {word} {random.choice(['X', 'Pro', 'Lite', 'Max', '2'])}"
    unit_cost = round(random.uniform(15, 900), 2)
    margin_pct = random.uniform(0.25, 0.70)  # margen bruto 25%-70%
    unit_price = round(unit_cost / (1 - margin_pct), 2)
    products_rows.append({
        "product_id": pid,
        "category_id": int(category["category_id"]),
        "name": name,
        "sku": f"NVT-{pid:04d}",
        "unit_price": unit_price,
        "unit_cost": unit_cost,
        "stock_quantity": random.randint(0, 500),
        "is_active": random.random() > 0.05,  # 5% de productos descatalogados
    })

df_products = pd.DataFrame(products_rows)
df_products.head()

,product_id,category_id,name,sku,unit_price,unit_cost,stock_quantity,is_active
0,1,8,Zenlo Cam Lite,NVT-0001,337.21,231.73,52,True
1,2,5,Kortex Note 2,NVT-0002,527.29,388.40,111,True
2,3,6,Zenlo Console 2,NVT-0003,417.37,190.97,279,True
3,4,5,Nyra Canvas X,NVT-0004,1012.37,686.54,216,True
4,5,8,Orbeon Lens Lite,NVT-0005,182.13,105.46,183,True


## 3. `customers`

In [5]:
customers_rows = []
for cid in range(1, N_CUSTOMERS + 1):
    country = random.choices(COUNTRIES, weights=COUNTRY_WEIGHTS, k=1)[0]
    first = fake.first_name()
    last = fake.last_name()
    reg_date = fake.date_between(start_date=ONE_YEAR_AGO - timedelta(days=365), end_date=ONE_YEAR_AGO)
    customers_rows.append({
        "customer_id": cid,
        "first_name": first,
        "last_name": last,
        "email": f"{first.lower()}.{last.lower()}{cid}@example.com",
        "country": country,
        "city": fake.city(),
        "acquisition_channel": random.choices(CHANNELS, weights=CHANNEL_WEIGHTS, k=1)[0],
        "registration_date": reg_date,
    })

df_customers = pd.DataFrame(customers_rows)
df_customers.head()

,customer_id,first_name,last_name,email,country,city,acquisition_channel,registration_date
0,1,Danielle,Johnson,danielle.johnson1@example.com,Spain,East Donald,paid_ads,2023-04-18
1,2,Curtis,Yang,curtis.yang2@example.com,Spain,New Roberttown,social_media,2023-08-01
2,3,Clayton,Hall,clayton.hall3@example.com,Portugal,New Jamesside,referral,2023-08-23
3,4,Robert,Cole,robert.cole4@example.com,Portugal,Port Lindachester,referral,2023-04-20
4,5,Matthew,Moore,matthew.moore5@example.com,Spain,Curtisfurt,referral,2023-10-25


## 4. `orders` + `order_items`

Cada pedido tiene entre 1 y 4 líneas; copio el precio/coste (snapshot) desde `products` en el momento de generar el pedido.

In [6]:
orders_rows = []
order_items_rows = []
order_item_id = 1

active_products = df_products[df_products["is_active"]].reset_index(drop=True)

for oid in range(1, N_ORDERS + 1):
    customer_id = random.randint(1, N_CUSTOMERS)
    order_date = fake.date_time_between(start_date=ONE_YEAR_AGO, end_date=TODAY)
    status = random.choices(ORDER_STATUSES, weights=ORDER_STATUS_WEIGHTS, k=1)[0]

    orders_rows.append({
        "order_id": oid,
        "customer_id": customer_id,
        "order_date": order_date,
        "status": status,
    })

    n_items = random.randint(1, 4)
    chosen_products = active_products.sample(n=n_items, replace=False)
    for _, prod in chosen_products.iterrows():
        order_items_rows.append({
            "order_item_id": order_item_id,
            "order_id": oid,
            "product_id": int(prod["product_id"]),
            "quantity": random.randint(1, 3),
            "unit_price": prod["unit_price"],
            "unit_cost": prod["unit_cost"],
        })
        order_item_id += 1

df_orders = pd.DataFrame(orders_rows)
df_order_items = pd.DataFrame(order_items_rows)
print(f"{len(df_orders)} pedidos, {len(df_order_items)} lineas de pedido")

2000 pedidos, 4990 lineas de pedido


## 5. `payments`

Genero un pago por pedido en el caso normal; simulo incidencias (pago fallido con reintento, reembolsos) para poder practicar detección de incidencias en la Parte 3.

In [7]:
payments_rows = []
payment_id = 1

order_totals = (
    df_order_items.assign(line_total=lambda d: d["quantity"] * d["unit_price"])
    .groupby("order_id")["line_total"].sum()
)

for _, order in df_orders.iterrows():
    oid = order["order_id"]
    total = round(float(order_totals.get(oid, 0.0)), 2)
    order_date = order["order_date"]

    if order["status"] == "cancelled":
        # pago fallido, sin reintento
        payments_rows.append({
            "payment_id": payment_id, "order_id": oid, "amount": total,
            "payment_date": order_date, "method": random.choice(PAYMENT_METHODS),
            "status": "failed",
        })
        payment_id += 1
        continue

    r = random.random()
    if r < 0.05:
        # intento fallido + reintento correcto
        payments_rows.append({
            "payment_id": payment_id, "order_id": oid, "amount": total,
            "payment_date": order_date, "method": random.choice(PAYMENT_METHODS),
            "status": "failed",
        })
        payment_id += 1
        payments_rows.append({
            "payment_id": payment_id, "order_id": oid, "amount": total,
            "payment_date": order_date + timedelta(minutes=random.randint(5, 120)),
            "method": random.choice(PAYMENT_METHODS), "status": "completed",
        })
        payment_id += 1
    elif r < 0.08 and order["status"] == "delivered":
        # pedido entregado pero luego reembolsado (devolucion)
        payments_rows.append({
            "payment_id": payment_id, "order_id": oid, "amount": total,
            "payment_date": order_date, "method": random.choice(PAYMENT_METHODS),
            "status": "refunded",
        })
        payment_id += 1
    else:
        payments_rows.append({
            "payment_id": payment_id, "order_id": oid, "amount": total,
            "payment_date": order_date, "method": random.choice(PAYMENT_METHODS),
            "status": "completed",
        })
        payment_id += 1

df_payments = pd.DataFrame(payments_rows)
print(f"{len(df_payments)} pagos generados")

2091 pagos generados


## 6. `shipments`

Genero envío solo para los pedidos `shipped` o `delivered`.

In [8]:
CARRIERS = ["SEUR", "Correos Express", "DHL", "GLS", "UPS"]
shipments_rows = []
shipment_id = 1

for _, order in df_orders.iterrows():
    if order["status"] not in ("shipped", "delivered"):
        continue
    shipped_date = order["order_date"] + timedelta(days=random.randint(1, 3))
    delivered_date = None
    ship_status = "in_transit"
    if order["status"] == "delivered":
        delivered_date = shipped_date + timedelta(days=random.randint(1, 6))
        ship_status = "delivered"

    shipments_rows.append({
        "shipment_id": shipment_id,
        "order_id": order["order_id"],
        "carrier": random.choice(CARRIERS),
        "status": ship_status,
        "shipped_date": shipped_date,
        "delivered_date": delivered_date,
    })
    shipment_id += 1

df_shipments = pd.DataFrame(shipments_rows)
print(f"{len(df_shipments)} envios generados")

1533 envios generados


## 7. `reviews`

Genero valoraciones solo para clientes con pedidos `delivered`, y no todos las tienen (~40%).

In [9]:
RATING_WEIGHTS = [0.04, 0.06, 0.15, 0.35, 0.40]  # sesgado a valoraciones altas
COMMENTS_POS = ["Muy buena calidad, volveria a comprar.", "Llego antes de lo esperado, encantado.",
                "Relacion calidad-precio excelente.", "Tal cual se describe, sin sorpresas."]
COMMENTS_NEG = ["No cumplio mis expectativas.", "Llego con un golpe en la caja.",
                "El producto tarda en cargar mas de lo anunciado.", "Atencion al cliente lenta."]

reviews_rows = []
review_id = 1

delivered_orders = df_orders[df_orders["status"] == "delivered"]
delivered_items = df_order_items[df_order_items["order_id"].isin(delivered_orders["order_id"])]

for _, item in delivered_items.iterrows():
    if random.random() > 0.40:
        continue
    order = delivered_orders[delivered_orders["order_id"] == item["order_id"]].iloc[0]
    rating = random.choices([1, 2, 3, 4, 5], weights=RATING_WEIGHTS, k=1)[0]
    comment = random.choice(COMMENTS_POS) if rating >= 4 else random.choice(COMMENTS_NEG)
    review_date = order["order_date"] + timedelta(days=random.randint(7, 30))

    reviews_rows.append({
        "review_id": review_id,
        "product_id": int(item["product_id"]),
        "customer_id": int(order["customer_id"]),
        "rating": rating,
        "comment": comment,
        "review_date": review_date.date(),
    })
    review_id += 1

df_reviews = pd.DataFrame(reviews_rows)
print(f"{len(df_reviews)} valoraciones generadas")

1341 valoraciones generadas


## 8. Cargar todo en BigQuery

Respeto el mismo orden que en `01_setup_bigquery.ipynb`.

In [10]:
TABLES_TO_LOAD = [
    ("categories", df_categories),
    ("customers", df_customers),
    ("products", df_products),
    ("orders", df_orders),
    ("order_items", df_order_items),
    ("payments", df_payments),
    ("shipments", df_shipments),
    ("reviews", df_reviews),
]

for table_name, df in TABLES_TO_LOAD:
    table_ref = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
    job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
    job.result()
    print(f"{table_name}: {len(df)} filas cargadas")

c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


categories: 8 filas cargadas


c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


customers: 500 filas cargadas


c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


...

payments: 2091 filas cargadas


c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


shipments: 1533 filas cargadas


c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


reviews: 1341 filas cargadas


## 9. Verificación rápida de recuento de filas

In [11]:
for table_name, df in TABLES_TO_LOAD:
    query = f"SELECT COUNT(*) AS n FROM `{PROJECT_ID}.{DATASET_ID}.{table_name}`"
    n_bq = client.query(query).to_dataframe()["n"].iloc[0]
    status = "OK" if n_bq == len(df) else "MISMATCH"
    print(f"{table_name:15s} local={len(df):6d}  bigquery={n_bq:6d}  [{status}]")

c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


categories      local=     8  bigquery=     8  [OK]


c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


customers       local=   500  bigquery=   500  [OK]


c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


...

payments        local=  2091  bigquery=  2091  [OK]


c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


shipments       local=  1533  bigquery=  1533  [OK]


c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


reviews         local=  1341  bigquery=  1341  [OK]
